# DSA Week 6 -- Heap & Priority Queue

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Prerequisites:** Weeks 1-5
**Focus:** heapq, top-k problems, priority scheduling

## Learning Objectives

1. Explain what a heap is and draw its tree structure
2. Use Python's `heapq` module for min-heap operations
3. Solve top-k problems efficiently with heaps
4. Build a priority queue for task scheduling
5. Benchmark heap-based top-k vs sorting

## The Big Idea

A heap is a binary tree where the parent is always smaller (min-heap) or larger
(max-heap) than its children. The key operations:

```
Min-Heap:
              1
            /   \
           3     5
          / \   /
         7   4 8

Properties:
  - Smallest element is always at the root -> O(1) to peek
  - Insert a new element -> O(log n)
  - Remove the smallest -> O(log n)
  - Build heap from list -> O(n)

Use cases:
  - Find top-k items without sorting everything
  - Priority queues (process highest-priority first)
  - Streaming data (maintain top-k as data arrives)
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Python's `heapq` Module

Python implements a min-heap using a regular list. The `heapq` module
provides functions to maintain the heap property.

In [ ]:
import heapq

# Build a heap from a list
data = [42, 15, 8, 23, 4, 16, 50, 31]
print("Original list: " + str(data))

heapq.heapify(data)  # O(n) -- rearranges in-place
print("After heapify: " + str(data))
print("Smallest:      " + str(data[0]) + " (always at index 0)")
print()

# Visualize the heap as a tree
def print_heap_tree(heap):
    """Print heap as ASCII tree."""
    if not heap:
        print("  (empty)")
        return
    n = len(heap)
    levels = []
    level_start = 0
    while level_start < n:
        level_end = min(level_start * 2 + 1, n) if level_start > 0 else 1
        if level_start == 0:
            level_end = 1
        else:
            level_end = min(level_start + 2 ** (len(levels)), n)
        # simpler approach: compute level
        pass

    # Simple level-by-level print
    import math
    depth = int(math.log2(n)) + 1 if n > 0 else 0
    idx = 0
    for level in range(depth):
        count = min(2 ** level, n - idx)
        indent = " " * (2 ** (depth - level) - 1)
        spacing = " " * (2 ** (depth - level + 1) - 1)
        values = []
        for i in range(count):
            if idx + i < n:
                values.append(str(heap[idx + i]).center(3))
        print(indent + spacing.join(values))
        idx += count

print("Heap as tree:")
print_heap_tree(data)

In [ ]:
# Core heap operations
import heapq

heap = []

# Push items
for val in [42, 15, 8, 23, 4]:
    heapq.heappush(heap, val)
    print("  push(" + str(val) + ") -> heap = " + str(heap) + " | min = " + str(heap[0]))

print()

# Pop items (always returns smallest)
print("Popping in order:")
while heap:
    val = heapq.heappop(heap)
    print("  pop -> " + str(val) + " | remaining = " + str(heap))

**Expected Output:**
```
  push(42) -> heap = [42] | min = 42
  push(15) -> heap = [15, 42] | min = 15
  push(8) -> heap = [8, 42, 15] | min = 8
  push(23) -> heap = [8, 23, 15, 42] | min = 8
  push(4) -> heap = [4, 23, 15, 42, 8] | min = 4

Popping in order:
  pop -> 4 | remaining = [8, 23, 15, 42]
  pop -> 8 | remaining = [15, 23, 42]
  pop -> 15 | remaining = [23, 42]
  pop -> 23 | remaining = [42]
  pop -> 42 | remaining = []
```

Notice: popping always gives elements in sorted order! This is why heaps
are used to implement priority queues.

---
## Part 2: The Top-K Problem

**Problem:** Find the k largest (or smallest) values in a dataset.

**Slow way:** Sort everything, take last k. O(n log n).
**Fast way:** Use a heap. O(n log k). When k is small, this is much faster!

```
Why heap is better for top-k:

  n = 1,000,000 and k = 10:
    Sorting:  O(n log n) = O(1,000,000 * 20) = 20,000,000 operations
    Heap:     O(n log k) = O(1,000,000 * 3)  = 3,000,000 operations
    Speedup:  ~7x

  n = 1,000,000 and k = 1:
    Sorting:  20,000,000 operations
    Heap:     1,000,000 operations (just one pass!)
    Or even simpler: just use max() -- O(n)
```

In [ ]:
import heapq
import random

random.seed(42)
readings = [round(random.gauss(50, 20), 2) for _ in range(100_000)]

# Method 1: Sort and slice (O(n log n))
top5_sort = sorted(readings, reverse=True)[:5]

# Method 2: heapq.nlargest (O(n log k))
top5_heap = heapq.nlargest(5, readings)

# Method 3: heapq.nsmallest
bot5_heap = heapq.nsmallest(5, readings)

print("Top 5 (sort):    " + str(top5_sort))
print("Top 5 (heap):    " + str(top5_heap))
print("Bottom 5 (heap): " + str(bot5_heap))
print()
print("Both methods give same result: " + str(top5_sort == top5_heap))

In [ ]:
# Benchmark: sort vs heap for top-k
import timeit

def top_k_sort(data, k):
    return sorted(data, reverse=True)[:k]

def top_k_heap(data, k):
    return heapq.nlargest(k, data)

print("=== Top-K Benchmark (n=100,000) ===")
print()
for k in [1, 5, 10, 100, 1000]:
    t_sort = timeit.timeit(lambda: top_k_sort(readings, k), number=20) / 20 * 1000
    t_heap = timeit.timeit(lambda: top_k_heap(readings, k), number=20) / 20 * 1000
    ratio = t_sort / t_heap if t_heap > 0 else 0
    print("  k=" + str(k).rjust(5) + ": sort=" + "{:.2f}ms".format(t_sort).rjust(10) + "  heap=" + "{:.2f}ms".format(t_heap).rjust(10) + "  ratio=" + "{:.1f}x".format(ratio).rjust(7))

print()
print("Note: heap advantage is biggest when k is small relative to n.")
print("When k approaches n, sorting is faster (heapq has higher constant).")

---
## Part 3: Priority Queue -- Process Tasks by Importance

A priority queue processes elements by priority, not by arrival order.
Lower number = higher priority (like severity levels: 1=critical, 5=low).

```
Priority Queue:
  push(3, "update docs")      -> queue: [(3, "update docs")]
  push(1, "fix sensor error") -> queue: [(1, "fix sensor"), (3, "update docs")]
  push(2, "generate report")  -> queue: [(1, "fix sensor"), (3, "update docs"), (2, "generate")]

  pop() -> (1, "fix sensor error")   -- highest priority processed first!
  pop() -> (2, "generate report")
  pop() -> (3, "update docs")
```

In [ ]:
import heapq

class PriorityQueue:
    """Priority queue using a min-heap."""

    def __init__(self):
        self.heap = []
        self.counter = 0  # tiebreaker for equal priorities

    def push(self, priority, item):
        """Add item with priority (lower = higher priority)."""
        heapq.heappush(self.heap, (priority, self.counter, item))
        self.counter += 1

    def pop(self):
        """Remove and return highest-priority item."""
        priority, _, item = heapq.heappop(self.heap)
        return priority, item

    def peek(self):
        """Look at next item without removing."""
        if self.heap:
            return self.heap[0][0], self.heap[0][2]
        return None

    def __len__(self):
        return len(self.heap)

    def is_empty(self):
        return len(self.heap) == 0


# Demo: sensor alert processing
pq = PriorityQueue()

alerts = [
    (3, "Low battery on sensor D03"),
    (1, "CRITICAL: Motor A overheating!"),
    (2, "Warning: Vibration above threshold"),
    (1, "CRITICAL: Communication lost with sensor D01"),
    (4, "Info: Scheduled maintenance due"),
    (2, "Warning: Humidity spike in room 204"),
]

print("=== Sensor Alert Priority Queue ===")
print()
print("Enqueueing alerts:")
for priority, msg in alerts:
    pq.push(priority, msg)
    print("  [" + str(priority) + "] " + msg)

print()
print("Processing by priority:")
while not pq.is_empty():
    priority, msg = pq.pop()
    print("  Processing [" + str(priority) + "] " + msg)

**Expected Output:**
```
=== Sensor Alert Priority Queue ===

Enqueueing alerts:
  [3] Low battery on sensor D03
  [1] CRITICAL: Motor A overheating!
  [2] Warning: Vibration above threshold
  [1] CRITICAL: Communication lost with sensor D01
  [4] Info: Scheduled maintenance due
  [2] Warning: Humidity spike in room 204

Processing by priority:
  Processing [1] CRITICAL: Motor A overheating!
  Processing [1] CRITICAL: Communication lost with sensor D01
  Processing [2] Warning: Vibration above threshold
  Processing [2] Warning: Humidity spike in room 204
  Processing [3] Low battery on sensor D03
  Processing [4] Info: Scheduled maintenance due
```

Critical alerts are processed first, regardless of when they arrived.

---
## When to Use Heaps

| Problem | Use Heap? | Why |
|---------|-----------|-----|
| Find top-k of n items (k << n) | YES | O(n log k) vs O(n log n) for sorting |
| Process tasks by priority | YES | Priority queue is THE use case |
| Maintain running top-k as data streams in | YES | Push + pop in O(log k) |
| Sort an entire list | NO | Just use `sorted()` -- Timsort is optimized |
| Find the single max or min | NO | Just use `max()` or `min()` -- O(n) |

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Forgetting heapq is a MIN-heap | For max-heap, negate values: `heappush(h, -val)` |
| Pushing non-comparable items | Wrap in tuple: `(priority, counter, item)` |
| Using heap when set would work | Heap for ordering, set for membership |

---
## Mini-Quiz

In [ ]:
# Q1: You have 10 million sensor readings. Find the 10 highest.
# What is the Big-O using: (a) sorting? (b) heap?
# Answer:

# Q2: Build a max-heap in Python (heapq only supports min-heap).
# Hint: negate the values.
import heapq
max_heap = []
for val in [3, 1, 4, 1, 5, 9]:
    heapq.heappush(max_heap, -val)  # negate!
# Pop max:
# max_val = -heapq.heappop(max_heap)
# print(max_val)  # should be 9

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)